In [ ]:
# data analysis
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from datetime import  datetime  

# data management
from ucimlrepo import fetch_ucirepo
from openmeteopy import OpenMeteo
from openmeteopy.hourly import HourlyHistorical
from openmeteopy.daily import DailyHistorical
from openmeteopy.options import HistoricalOptions
from openmeteopy.utils.constants import *
import pickle

# models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# system 
import os

# global parameters
name_data_folder = "data"
name_uci_raw = "raw_EP"
name_meteo_raw = 'raw_METEO'
name_combined_raw = 'raw_combined'

# Part 1 - download data

We download data that contains
- energy usage of `UCL` 
- weather data from UCI Machine Learning Repository
for localization: `Sceaux Ile-de-France` ($48.77^\circ\mathrm{N}, 2.29^\circ\mathrm{E}$)

Methodology
- If data does not exits, download `UCL data` using `ucimlrepo` library, saving it using `pickle` library
- If data exits, load data
- Date, time format: we use `iso8601` (YYYY-MM-DDTHH:mm:ss±hh:mm)

About data:
- raw_EP
    1. **date: Date in format dd/mm/yyyy**
    2. **time: time in format hh:mm:ss**
    3. **global_active_power: household global minute-averaged active power (in kilowatt)**
        > Comment: This power is *consumed* by circuit - it does perform work - **Explained variable**
    4. global_reactive_power: household global minute-averaged reactive power (in kilowatt)
        > Comment: This power is *stored* in circuit - it does not perform work but it is necessary for infrastructure (transformers etc.)
    5. voltage: minute-averaged voltage (in volt)
    6. global_intensity: household global minute-averaged current intensity (in ampere)
    7. sub_metering_1: energy sub-metering No. 1 (in watt-hour of active energy). It corresponds to  the kitchen, containing mainly a dishwasher, an oven and a microwave (hot plates are not el ectric but gas powered).
    8. sub_metering_2: energy sub-metering No. 2 (in watt-hour of active energy). It corresponds to  the laundry room, containing a washing-machine, a tumble-drier, a refrigerator and a li ght.
    9. sub_metering_3: energy sub-metering No. 3 (in watt-hour of active energy). It corresponds to an electric water-heater and an air-conditioner.
- raw_METEO 
    1. **temperature_2m: Air temperature measured at 2 meters above ground (°C)**  
        > Comment: Directly affects heating/cooling demand — highly relevant.
    2. **relativehumidity_2m: Relative humidity at 2 meters (%)**  
        > Comment: Influences comfort and cooling demand — relevant.
    3. **dewpoint_2m: Dew point temperature at 2 meters (°C)**  
        > Comment: Related to humidity and cooling needs — relevant.
    4. **apparent_temperature: Perceived temperature considering wind and humidity (°C)**  
        > Comment: Strong proxy for human comfort — relevant.
    5. pressure_msl: Atmospheric pressure at mean sea level (hPa)
        > Comment: Weather indicator, weak direct link to electricity use.
    6. surface_pressure: Atmospheric pressure at surface level (hPa)
        > Comment: Similar to above, weak relevance.
    7. **cloudcover: Total cloud cover (%)**  
        > Comment: Affects solar radiation and daylight — relevant.
    8. **cloudcover_low: Low-level cloud cover (%)**  
        > Comment: Clouds on different levels differs in density - absorption - relevant.
    9. **cloudcover_mid: Mid-level cloud cover (%)**  
        > Comment: Same as above.
    10. **cloudcover_high: High-level cloud cover (%)**  
        > Comment: Same as above.
    11. **windspeed_10m: Wind speed at 10 meters (m/s)**  
        > Comment: Can affect cooling/heating indirectly — moderate relevance.
    12. windspeed_100m: Wind speed at 100 meters (m/s)
        > Comment: More relevant for wind energy production, not household demand.
    13. winddirection_10m: Wind direction at 10 meters (°)
        > Comment: Very weak link to electricity consumption.
    14. winddirection_100m: Wind direction at 100 meters (°)
        > Comment: Same as above.
    15. **windgusts_10m: Maximum wind gusts at 10 meters (m/s)**  
        > Comment: Weather temperature translation indicator, moderate link to demand.
    16. **shortwave_radiation: Incoming shortwave solar radiation (W/m²)**  
        > Comment: Strongly affects daylight and solar heating — relevant.
    17. **direct_radiation: Direct solar radiation (W/m²)**  
        > Comment: Relevant for solar exposure and PV production.
    18. **diffuse_radiation: Diffuse solar radiation (W/m²)**  
        > Comment: Same as above.
    19. **vapor_pressure_deficit: Difference between saturation and actual vapor pressure (kPa)**  
        > Comment: Technical humidity measure, indirect relevance.
    20. et0_fao_evapotranspiration: Reference evapotranspiration (mm/day) 
        > Comment: Agricultural metric, weak relevance for household electricity.
    21. **precipitation: Total precipitation (mm)**  
        > Comment: Rainfall affects daylight and comfort — moderate relevance.
    22. **weathercode: Encoded weather condition (categorical)**  
        > Comment: Useful categorical proxy for overall weather — relevant.
    23. **snowfall: Snowfall amount (mm)**  
        > Comment: Strongly affects heating demand — relevant.
    24. soil_moisture_0_to_7cm: Soil moisture in top layer (m³/m³)
        > Comment: Agricultural metric, no direct link to household demand.
    25. soil_moisture_7_to_28cm: Soil moisture (m³/m³) 
        > Comment: Same as above.
    26. soil_moisture_28_to_100cm: Soil moisture (m³/m³)  
        > Comment: Same as above.
    27. soil_moisture_100_to_255cm: Soil moisture (m³/m³)  
        > Comment: Same as above.
    28. soil_temperature_0_to_7cm: Soil temperature (°C)  
        > Comment: Agricultural relevance, weak link to household demand.
    29. soil_temperature_7_to_28cm: Soil temperature (°C)  
        > Comment: Same as above.
    30. soil_temperature_28_to_100cm: Soil temperature (°C)  
        > Comment: Same as above.
    31. soil_temperature_100_to_255cm: Soil temperature (°C)  
        > Comment: Same as above.

Where, **bold** variables are relevant for ous

In [ ]:
### ----------- UCI ----------- ###

# flag 
exists = any(filename.startswith(name_uci_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_uci_raw} already exists')
    print(f"{name_uci_raw} Data loading...")
    ## opening file
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", "rb") as file:
        ## loading file
        df_raw_EP = pickle.load(file)
    print(f"Done")
# download data
else:
    print(f"Downloading {name_uci_raw} data ...")
    ## downloading data
    df_raw_EP = fetch_ucirepo(id=235)       
    print(f"Download complete \nSaving data...") 
    ## creating file for writing uci repo
    with open(f"{name_data_folder}/{name_uci_raw}.pkl", 'wb') as file:
        ## saving file
        pickle.dump(df_raw_EP, file)
    print('Done')

# leak check 
num_instances_metadata = df_raw_EP.metadata['num_instances']
num_intances_local = df_raw_EP.data.features.shape[0]
if num_intances_local != num_instances_metadata:
    print(f'''There is leak of data in your data set:
          - global data instances = {num_instances_metadata}
          - local data instances = {num_intances_local}
        Removing local dataset ...
          ''')
    os.remove(f"{name_data_folder}/{name_uci_raw}")
    print('Local dataset removed\nRun cell again')
else:
    print(f"\n---There is no leak in data---\n")

# load UCI data into df
df_EP = df_raw_EP.data.features
# variable information 
print(df_raw_EP.variables)

In [ ]:
### ----------- JOIN DATASETS I ----------- ###

# create final df
## UCI DATA
df_EP_copy = df_EP.copy()
### update `Date` and `Time` column to `datetime` type
df_EP_copy['Date'] = df_EP_copy['Date'] + ' ' + df_EP_copy['Time']
df_EP_copy['Date'] = pd.to_datetime(df_EP_copy['Date'], format='%d/%m/%Y %H:%M:%S', utc=True)
df_EP_copy = df_EP_copy.drop('Time', axis=1)
### set time index
df_EP_copy = df_EP_copy.set_index('Date')
### set float64 data type for rest columns 
df_EP_copy  = df_EP_copy.apply(pd.to_numeric, errors="coerce")
### We need to change Date in out `df` file - we need to get rid off minutes
df_EP_copy = df_EP_copy.resample('H').mean()
df_EP_copy

In [ ]:
### ----------- METEO ----------- ###

# parameters - Sceux Ile de France
longitude = 48.7764 # 48.66 N
latitude =  2.2903 # 2.29 E 

exists = any(filename.startswith(name_meteo_raw) for filename in os.listdir(name_data_folder))

# load data
if exists:
    print(f'{name_meteo_raw} already exists')
    print(f"{name_meteo_raw} Data loading...")
    ## open file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "rb") as file:
        ## load data
        df_raw_meteo = pickle.load(file)
    print(f"Done")
# download data 
else:
    try:
        start_date, end_date = str(df_EP_copy.index.date.min()), str(df_EP_copy.index.date.max())
    except:
        raise 'To download name_meteo_raw, you need run earlier cells'
    print(f"Downloading {name_meteo_raw} data ...")
    ## set download options
    hourly = HourlyHistorical()
    daily = DailyHistorical()
    options = HistoricalOptions(latitude,
                                longitude, 
                                start_date=start_date,
                                end_date=end_date)
    ## download 
    ##TODO - check if we can skip minutes (redundant)
    df_raw_meteo = OpenMeteo(options, hourly.all(), daily.all())
    print(f"Download complete \nSaving data...") 
    ## saving file
    with open(f"{name_data_folder}/{name_meteo_raw}.pkl", "wb") as file:
        pickle.dump(df_raw_meteo, file)
    print('Done')
# Download data
df_meteo = df_raw_meteo.get_pandas()[0]
df_meteo

In [ ]:
### ----------- JOIN DATASETS II ----------- ###

## METEO DATA
df_meteo_copy = df_meteo.copy()
df_meteo_copy = df_meteo_copy.rename_axis('Date')
df_meteo_copy.index = pd.to_datetime(df_meteo_copy.index, format="%Y-%m-%dT%H:%M", utc=True)
## MERGE DATA
### joining 
df = df_EP_copy
df = df.join(df_meteo_copy, how='outer')
df
### saving data
df.to_json(f"{name_data_folder}/{name_combined_raw}.json")

# Part 2 - data engineering 

Methodology 
- Reduce meaningless data
    - We reduce features from `raw_METEO` that are meaningless
    - Selected features are **bolded** in [Part 1 - raw_METEO](#Part-1---download-data)
- Variable selection:
    - explained column: `Global_active_power` from `raw_EP`
    - explainable columns: Columns from `raw_METEO`
- Resampling data: *YYYY/MM/DD h* $\rightarrow$ *YYYY/MM/DD*
    - we aggregate by *mean*
- Remove missing rows
    - We remove all rows with NaN values
- Splitting data into two different `DataFrames` objects:
    - explained data $\rightarrow$ `y`
    - explainable data $\rightarrow$ `X`
- Feature engineering 
    - TDO Given variables, colinearity reduction
    - PCR reduction
        - Centralization and normalization: (Meeting theorem assumption)
        - PCR reduction with explainability(? - misspelled ?) threshold at $0.95\%$ 
    - TODO Partial least square regression

In [ ]:
### ----------- data preparation ----------- ###

# loading data
df = pd.read_json(f"{name_data_folder}/{name_combined_raw}.json")

# parameters 
## target & features
explained_column = 'Global_active_power'
explainable_columns = [
    "temperature_2m",        # 1
    "relativehumidity_2m",   # 2
    "dewpoint_2m",           # 3
    "apparent_temperature",  # 4
    "cloudcover",            # 7
    "cloudcover_low",        # 8
    "cloudcover_mid",        # 9
    "cloudcover_high",       # 10
    "windspeed_10m",         # 11
    "windgusts_10m",         # 15
    "shortwave_radiation",   # 16
    "direct_radiation",      # 17
    "diffuse_radiation",     # 18
    "vapor_pressure_deficit",# 19
    "precipitation",         # 21
    "weathercode",           # 22
    "snowfall"               # 23
]
## Data selection params:
pca_explainability = 0.95
## train_test split params:
param_train_size = 0.8

# TODO - criteria selection
print('Columns selection ...\n')
## select columns
mask_columns = [explained_column] + explainable_columns
## reduce df  
df = df[mask_columns]

# aggregate datetime data to day by mean
print('Resampling data ...\n')
df = df.resample('d').mean()

# remove missing rows
mask_notna = df.notna().all(axis=1)
print(f'Removing missing rows...\n-from {df.shape[0]} rows {df.shape[0]-mask_notna.sum()} were removed\n')
df = df[mask_notna]

# split df to explained (`y`) and explanatory (`X`) variables
y = df[[explained_column]]
X = df.drop(columns=explained_column)
n, m = X.shape
print(f"X have shape {n}x{m}")

Feature engineering 

In [ ]:
### ----------- data selection - standard ----------- ###

## standard variables 
### TODO - vifs analysis and reduction

In [ ]:
### ----------- data selection - PCR ----------- ###

## TODO PCR reduction
print("PCR ...")
### standardization 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
### PCR reduction 
pca = PCA(n_components=pca_explainability)
X_pca = pca.fit_transform(X_scaled)
### Recreate Dataframe with same indeks 
X_selected = pd.DataFrame(
    X_pca,
    index = X.index,
    columns = [f"PC{i+1} - {var}" for i, var in zip(range(X_pca.shape[1]), pca.explained_variance_ratio_)]
)
print(f"Results:\nFeatures: {X.shape[1]} -> {X_selected.shape[1]}\nExplainability:\n{pca.explained_variance_ratio_}")

In [ ]:
### ----------- data selection - partial least squares ----------- ###

## TODO Patrial least square regression 
### ...

Splitting data

In [ ]:
# Split data


X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, 
    test_size=1-param_train_size, train_size=param_train_size
)

# check sizes
if X_train.shape[0] != y_train.shape[0]:
    raise 'Dimensions do not match'


# Part 2 - data engineering 

Have fun :)
- You only need to work no X_train, X_test, y_train, y_test, 
- 